<a href="https://colab.research.google.com/github/perfectmakuwerere/efficient-llm-lab/blob/main/llm_compressor_script(GPTQModifier%2CSmoothQuantModifier).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


%%writefile compress.py

import os
import gc
import argparse
import torch

BASE_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

VARIANTS = {
    "w4a16-wiki": {
        "output_dir": "./SmolLM2-1.7B-W4A16-wiki",
        "dataset": "wikitext",
        "dataset_config_name": "wikitext-2-raw-v1",
        "dataset_split": "train",
        "recipe": "w4a16",
    },
    "w4a16-instruct": {
        "output_dir": "./SmolLM2-1.7B-W4A16-instruct",
        "dataset": "HuggingFaceH4/ultrachat_200k",
        "dataset_config_name": None,
        "dataset_split": "train_sft",
        "recipe": "w4a16",
    },
    "w8a8-instruct": {
        "output_dir": "./SmolLM2-1.7B-W8A8-instruct",
        "dataset": "HuggingFaceH4/ultrachat_200k",
        "dataset_config_name": None,
        "dataset_split": "train_sft",
        "recipe": "w8a8",
    },
}

MAX_SEQ_LENGTH = 2048
NUM_CALIBRATION_SAMPLES = 256


def build_recipe(recipe_name: str):
    """Return the llm-compressor recipe for the given variant name."""
    from llmcompressor.modifiers.quantization import GPTQModifier
    from llmcompressor.modifiers.transform import SmoothQuantModifier

    if recipe_name == "w4a16":
        return GPTQModifier(
            scheme="W4A16",
            targets="Linear",
            ignore=["lm_head"],
        )
    elif recipe_name == "w8a8":
        # SmoothQuant first smooths activation outliers, then GPTQ quantizes
        # both weights and activations to 8-bit. The combo is standard for W8A8.
        return [
            SmoothQuantModifier(smoothing_strength=0.8),
            GPTQModifier(
                scheme="W8A8",
                targets="Linear",
                ignore=["lm_head"],
            ),
        ]
    else:
        raise ValueError(f"Unknown recipe: {recipe_name}")


def load_calibration_dataset(config: dict):
    """
    Load and preprocess the calibration dataset.

    llm-compressor expects a dataset with a plain 'text' column.
    Ultrachat stores conversations as message dicts, so we flatten
    them to plain text. Wikitext already has a 'text' column.
    """
    from datasets import load_dataset

    dataset = load_dataset(
        config["dataset"],
        name=config.get("dataset_config_name"),
        split=config.get("dataset_split", "train"),
    )

    # If dataset already has a text column, use it as-is
    if "text" in dataset.column_names:
        return dataset

    # Ultrachat format: messages is a list of {"role": ..., "content": ...} dicts
    # Flatten each conversation into plain text for the tokenizer
    def format_conversation(example):
        text = "\n".join(
            f"{msg['role'].upper()}: {msg['content']}"
            for msg in example["messages"]
        )
        return {"text": text}

    dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)
    return dataset


def compress_variant(name: str, config: dict):
    """Run oneshot compression for a single variant."""
    from llmcompressor import oneshot

    output_dir = config["output_dir"]

    # Skip if already compressed successfully
    if os.path.exists(os.path.join(output_dir, "config.json")):
        print(f"\n[{name}] Already exists at {output_dir}, skipping.")
        return

    print(f"\n{'='*60}")
    print(f"[{name}] Starting compression")
    print(f"  Base model : {BASE_MODEL}")
    print(f"  Dataset    : {config['dataset']} / {config.get('dataset_split', 'train')}")
    print(f"  Output     : {output_dir}")
    print(f"{'='*60}")

    recipe = build_recipe(config["recipe"])
    dataset = load_calibration_dataset(config)

    oneshot(
        model=BASE_MODEL,
        dataset=dataset,
        recipe=recipe,
        output_dir=output_dir,
        max_seq_length=MAX_SEQ_LENGTH,
        num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    )

    print(f"[{name}] Done. Saved to {output_dir}")

    # Free VRAM between runs — critical on Colab
    gc.collect()
    torch.cuda.empty_cache()


def print_size_summary():
    """Print a size comparison table across all variants after compression."""
    import pathlib

    def dir_size_gb(path):
        p = pathlib.Path(path)
        if not p.exists():
            return None
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

    print("\n" + "="*50)
    print(f"{'Variant':<25} {'Size (GB)':>10} {'vs bf16':>10}")
    print("-"*50)

    bf16_size = 3.4

    rows = [("bf16 baseline", bf16_size, None)]
    for name, cfg in VARIANTS.items():
        size = dir_size_gb(cfg["output_dir"])
        if size is not None:
            pct = (size / bf16_size - 1) * 100
            rows.append((name, size, pct))

    for variant, size, pct in rows:
        if pct is None:
            print(f"{variant:<25} {size:>9.2f}G {'—':>10}")
        else:
            print(f"{variant:<25} {size:>9.2f}G {pct:>+9.1f}%")

    print("="*50)


def main():
    parser = argparse.ArgumentParser(description="Compress SmolLM2-1.7B-Instruct variants")
    parser.add_argument(
        "--variant",
        choices=list(VARIANTS.keys()) + ["all"],
        default="all",
        help="Which variant to compress (default: all)",
    )
    args = parser.parse_args()

    targets = VARIANTS if args.variant == "all" else {args.variant: VARIANTS[args.variant]}

    print(f"Compressing {len(targets)} variant(s) of {BASE_MODEL}")
    print(f"Calibration samples : {NUM_CALIBRATION_SAMPLES}")
    print(f"Max seq length      : {MAX_SEQ_LENGTH}")

    for name, config in targets.items():
        compress_variant(name, config)

    print_size_summary()
    print("\nAll done. Next step: run evaluate.py")


if __name__ == "__main__":
    main()


Overwriting compress.py


In [ ]:
# 2. confirm the fix took effect
!grep "ultrachat" compress.py

        "dataset": "HuggingFaceH4/ultrachat-200k",
        "dataset": "HuggingFaceH4/ultrachat-200k",


In [ ]:
!python compress.py --variant w4a16-instruct

Compressing 1 variant(s) of HuggingFaceTB/SmolLM2-1.7B-Instruct
Calibration samples : 256
Max seq length      : 2048
2026-06-11 18:56:13.132402: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-11 18:56:13.201949: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

[w4a16-instruct] Starting compression
  Base model : HuggingFaceTB/SmolLM2-1.7B-Instruct
  Dataset    : HuggingFaceH4/ultrachat_200k / train_sft
  Output     : ./SmolLM2-1.7B-W4A16-instruct
Map: 100% 207865/207865 [00:20<00:00, 10078.55 examples/s]
2026-06

In [ ]:
!python compress.py --variant w8a8-instruct